In [0]:
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.types import IntegerType, StringType, LongType
import datetime
import json

# ---------- Helper functions ----------
def parse_time(hhmm):
    try:
        hhmm_str = str(int(hhmm)).zfill(4)
        return f"{hhmm_str[:2]}:{hhmm_str[2:]}"
    except:
        return None

def calculate_actual_day(day, week):
    try:
        return int(day) - (int(week) - 1) * 7
    except:
        return None

def get_weekday(day_num):
    weekday_map = {
        1: ("Sunday", 1),
        2: ("Monday", 2),
        3: ("Tuesday", 3),
        4: ("Wednesday", 4),
        5: ("Thursday", 5),
        6: ("Friday", 6),
        7: ("Saturday", 7)
    }
    return weekday_map.get(day_num, (None, None))

# ---------- Register UDFs ----------
parse_time_udf = udf(parse_time, StringType())
actual_day_udf = udf(calculate_actual_day, IntegerType())
weekday_name_udf = udf(lambda d: get_weekday(d)[0], StringType())
weekday_number_udf = udf(lambda d: get_weekday(d)[1], IntegerType())

# ---------- Load and process ----------
df = spark.read.format("delta").table("bronze.sales")

# Cast columns
df = df.withColumn("code", col("code").cast(LongType()))
int_cols = [col(c).cast("int").alias(c) for c in df.columns if c not in ["code"] and not c.startswith("etl_record_")]
df = df.select("code", *int_cols)

# Filter out rows with null code
df = df.filter(col("code").isNotNull())

# Derived columns
df = df.withColumn("time_formatted", parse_time_udf(col("time")))
df = df.withColumn("actual_day", actual_day_udf(col("day"), col("week")))
df = df.withColumn("weekday_name", weekday_name_udf(col("actual_day")))
df = df.withColumn("weekday_number", weekday_number_udf(col("actual_day")))

# Audit timestamps
now_utc = datetime.datetime.utcnow()
df = df.withColumn("etl_record_created_date", lit(now_utc))
df = df.withColumn("etl_record_modified_date", lit(now_utc))

# ---------- Write to Unity Catalog ----------
catalog_name = "db_dataclassdev"
schema_name = "silver"
table_name = "sales"
qualified_table_name = f"{catalog_name}.{schema_name}.{table_name}"

silver_path = f"abfss://silver@strdatabrickssadls.dfs.core.windows.net/{table_name}"

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", silver_path) \
    .saveAsTable(qualified_table_name)

# ---------- Return audit info ----------
audit_info = {
    "fileName": table_name,
    "rowCount": df.count(),
    "status": "Succeeded",
    "destinationPath": silver_path,
    "ucTable": qualified_table_name,
    "timestamp": str(now_utc)
}

dbutils.notebook.exit(json.dumps(audit_info))
